# Sensitivity Analysis: RAG Prompt Footprint & Retrieval Empirics
We run all 20 benchmark queries from the Ground Truth set directly against the RAG microservice (/recommend), bypassing the LLM generation, to empirically measure:

- **Prompt footprint:** Number of tokens in the generated prompt/context block per query (as would be sent to the LLM)
- **Retrieval breadth:** Number of courses retrieved or planned per query

For each query, we record:
1. The exact query text and user ID
2. The number of tokens in the RAG-generated context (using the LLM’s tokenizer, gpt2)

## Interpretation 
- **Prompt reduction:** Most queries send <500 tokens of context to the LLM—about a 10x reduction compared to dumping the full course catalog (~12,600 tokens).
- **Out-of-scope efficiency:** Fallback/irrelevant queries (Q18–Q20) yield 0-token prompts, showing the pipeline exits early and efficiently.* 
- **Complex queries:** Multi-semester planning or “double-dip” constraints can push prompts up to ~1300 tokens, so context management is necessary for models with 1k–2k window limits.



In [ ]:
# 1. Imports and tokenizer setup
import pandas as pd
import requests
import json
import numpy as np
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("gpt2")

GROUND_TRUTH_PATH = "Ground_Truth.xlsx"
RAG_ENDPOINT = "http://localhost:8001/recommend"


c:\Users\roblo\anaconda3\envs\deep_learning_0401\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\roblo\anaconda3\envs\deep_learning_0401\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\roblo\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activ

In [2]:
# 2. Helper function for token counting
def tokens_in_text(text):
    return len(tokenizer.encode(text))


In [3]:
# 3. Load the Ground Truth Excel file
df = pd.read_excel(GROUND_TRUTH_PATH)
df.head()


,case_id,user_id,query,llm_response,long_term
0,1,6362138,"{\n ""q"":\t""What courses should I take next ...","{\n ""thought"": """",\n ""advice"": ""As your ...",0.0
1,2,6362138,"{\n ""q"":\t""Plan the rest of my CS-BS degree...","{\n ""thought"": """",\n ""advice"": ""As your ...",1.0
2,3,6362138,"{\n ""q"":\t""I’d like an AI-oriented schedule...","{\n ""thought"": """",\n ""advice"": ""As your ...",0.0
3,4,6362138,"{\n ""q"":\t""Suggest five courses for Spring ...","{\n ""thought"": """",\n ""advice"": ""As your ...",0.0
4,5,6362138,"{\n ""q"":\t""Plan my final academic year so I...","{\n ""thought"": """",\n ""advice"": ""As your ...",1.0


In [5]:
# 4. Send each query to the RAG endpoint and analyze responses

tokens_per_prompt = []
n_retrieved = []
query_texts = []

for i, row in df.iterrows():
    user_id = int(row['user_id'])
    q_raw = row['query']
    try:
        # Extract question from JSON string or dict
        q = eval(q_raw) if isinstance(q_raw, str) else q_raw
        if isinstance(q, dict):
            question = q.get("q", str(q))
        else:
            question = str(q)
    except Exception:
        question = str(q_raw)

    payload = {
        "q": question,
        "user_id": user_id
        
    }
    try:
        response = requests.post(RAG_ENDPOINT, json=payload, timeout=20)
        if response.status_code != 200:
            print(f"Error for Q{i+1}: {response.status_code} {response.text}")
            continue
        rag = response.json()
    except Exception as e:
        print(f"Error for Q{i+1}: {e}")
        continue

    prompt = rag.get("prompt", "")
    tokens = tokens_in_text(prompt)
    tokens_per_prompt.append(tokens)

    # Number of recommended courses: from 'course_ids' (short-term) or 'plan' (long-term)
    if "course_ids" in rag:
        n_courses = len(rag["course_ids"])
    elif "plan" in rag and isinstance(rag["plan"], list):
        n_courses = sum(len(x["courses"]) for x in rag["plan"])
    else:
        n_courses = 0
    n_retrieved.append(n_courses)
    query_texts.append(question)

    print(f"Q{i+1}: {tokens} tokens, {n_courses} courses, Query: {question}")


Q1: 509 tokens, 6 courses, Query: What courses should I take next semester to stay on track for my CS-BS degree (max 15 credits)?
Q2: 342 tokens, 12 courses, Query: Plan the rest of my CS-BS degree so I can graduate on time.
Q3: 479 tokens, 5 courses, Query: I’d like an AI-oriented schedule for next Fall, max 12 credits. What should I take?
Q4: 457 tokens, 5 courses, Query: Suggest five courses for Spring that move me closest to graduation.
Q5: 340 tokens, 12 courses, Query: Plan my final academic year so I can graduate next Summer 2026.
Q6: 423 tokens, 4 courses, Query: Aurora, can you build a balanced 9-credit schedule for next semester?


Token indices sequence length is longer than the specified maximum sequence length for this model (1341 > 1024). Running this sequence through the model will result in indexing errors


Q7: 1341 tokens, 24 courses, Query: Are there any courses I can take that will help me fulfill two areas at once (“double dip”)?
Q8: 1346 tokens, 24 courses, Query: I am feeling overwhelmed by courses and it's affecting my mental health. What are my options and resources now and how can I build a more balanced schedule next semester?
Q9: 1318 tokens, 24 courses, Query: Which classes do I need to take for this major?
Q10: 1324 tokens, 24 courses, Query: What classes should I take if I want to focus on Software Engineering?
Q11: 382 tokens, 4 courses, Query: Please suggest my next term schedule to finish the CS minor (12-credit cap).
Q12: 259 tokens, 2 courses, Query: If I can only take 6 credits next term, what should I enroll in to progress on the CS minor?
Q13: 250 tokens, 2 courses, Query: Recommend electives that match human–computer interaction interests once I finish the required minor cores.
Q14: 215 tokens, 1 courses, Query: I need to take only 1 course for Summer… which one do 

In [6]:
# 5. Summary statistics
print("\n===== SUMMARY =====")
print(f"Total queries: {len(tokens_per_prompt)}")
print(f"Average tokens per prompt: {np.mean(tokens_per_prompt):.1f} ± {np.std(tokens_per_prompt):.1f}")
print(f"Average recommended courses per query: {np.mean(n_retrieved):.1f} ± {np.std(n_retrieved):.1f}")

print("\nMin/max tokens: ", min(tokens_per_prompt), "/", max(tokens_per_prompt))
print("Min/max recommended courses: ", min(n_retrieved), "/", max(n_retrieved))



===== SUMMARY =====
Total queries: 20
Average tokens per prompt: 492.1 ± 443.9
Average recommended courses per query: 7.8 ± 8.7

Min/max tokens:  0 / 1346
Min/max recommended courses:  0 / 24
